In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

In [2]:
# --- IMPORTING FROM OTHER FILES
import sys
import os

# Add path to the package: relativistic_dof
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../..")))
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../../..")))
from relativistic_dof import RelativisticDOFRegistry
from axion_model_analysis import AxionModelDistribution
from maxim_distribution_analysis import AxionModelMaximDistribution
from second_interpolation import get_process

## Configure the Setup and Select the Production Channel

In this section, we choose the axion production process to analyze and set flags that control output behavior (e.g., saving plots or data). Additional parameters:

- `file_number`: We define it, to select the process:
  - $0$ : Electron scattering
  - $1$ : Muon decay
  - $2$ : Muon scattering
  - $3$ : Tau decay
  - $4$ : Tau scattering

In [3]:
# --- SELECT FILE, SET FLAGS (MANUAL) ---
file_number = 4
x_decouple = 30
flag_save_plot_comparison = True

In [4]:
# --- Define process labels and corresponding filenames for selected production channel ---
title_arr = [
    r"$\bf e$ scattering",
    r"$\bf \mu$ decay", 
    r"$\bf \mu$ scattering",
    r"$\bf \tau$ decay", 
    r"$\bf \tau$ scattering"
]
filename_dist_arr = [
    "Distributions_fa_e_scat.dat", 
    "Distributions_fa_mu_dec.dat", 
    "Distributions_fa_mu_scat.dat",
    "Distributions_fa_tau_dec_extended.dat", #"Distributions_fa_tau_dec.dat" 
    "Distributions_fa_tau_scat_extended.dat", #"Distributions_fa_tau_scat.dat"
]
proces_name_arr = [
    "electron_scattering", 
    "muon_decay", 
    "muon_scattering", 
    "tau_decay", 
    "tau_scattering"
]

# --- Define particle masses [in eV] ---
electron_mass = 0.511e6      # [eV]
muon_mass = 105.66e6         # [eV]
tau_mass = 1777e6            # [eV]
particle_mass_arr = [electron_mass, muon_mass, muon_mass, tau_mass, tau_mass]
####################################################################################################################
filename_dist = f"../../data-new/{filename_dist_arr[file_number]}"
# filename_dist = f"/home/krzysztof/Workspace/Master-coding/real_distribution/data-new/{filename_dist_arr[file_number]}"
proces_name = f"{proces_name_arr[file_number]}"
partilce_mass = particle_mass_arr[file_number]

In [5]:
# --- Initialize axion model and extract distribution-based results ---

# Create an instance of the axion model using:
# - the selected distribution file,
# - the mass of the interacting particle,
# - the chosen decoupling point (x = m/T)
axionModel = AxionModelMaximDistribution(
    filename_dist,
    partilce_mass,
    x_decouple
)

# Compute and retrieve ΔN_eff and axion abundance as functions of fa
our_data = axionModel.get_neff_and_ya_vs_fa()

# --- Initialize axion model and extract second interpolation-based results ---

# Create an instance of the axion model using:
# - the selected distribution file,
# - the mass of the interacting particle,
axionModelInt = AxionModelDistribution(
    proces_name, 
    filename_dist, 
    partilce_mass
)

# Compute and retrieve ΔN_eff and axion abundance as functions of fa
our_data_int = axionModelInt.get_neff_and_ya_vs_fa()

## Sanity Check — Compare Our Results to Maxim's Data

In this section, we verify that our implementation correctly reproduces the reference results from Maxim's data files. In particular, we compare the calculated values of  $\Delta N_{\text{eff}}$ using our integration of his distribution with the original values reported in the files. This is an important consistency check to ensure that:
- We’re interpreting and integrating the distribution data correctly,
- The numerical methods are properly implemented,
- Any observed differences are physical and not due to a bug or misalignment.

We will visualize the comparison in the plot below.


In [6]:
# --- Plot ΔN_eff: Our Results vs Maxim's Reference Data ---

fig, axs = plt.subplots(figsize=(8, 6))

# --- Plot Our ΔN_eff results (from integration using Maxim's distribution) ---
axs.plot(our_data["fa"], our_data["delta_neff"], color='red', lw=3, linestyle='--',
         label=r'$\Delta N_{\rm eff}$: Our, fBE (integration of distributions)')

# --- Plot Our ΔN_eff results (from second interpolation using Maxim's distribution) ---
axs.plot(our_data_int["fa"], our_data_int["delta_neff"], color='blue', lw=3, linestyle='--',
         label=r'$\Delta N_{\rm eff}$: Our, fBE (second interpolation)')


# --- Axis scaling and labels ---
axs.set_xscale('log')
axs.set_xlabel(r'Decay Constant $\bf f_a$ [GeV]', fontsize=14, fontweight='bold')
axs.set_ylabel(r'$\bf \Delta N_{\rm eff}$', fontsize=14, fontweight='bold')
axs.set_title(title_arr[file_number], fontsize=16, fontweight='bold')

# --- Grid and legend ---
axs.grid(True, linestyle='--', linewidth=0.5)
axs.legend(fontsize=12)

# --- Planck 2018 constraint line and shaded exclusion area ---
planck_limit = 0.33
axs.axhline(planck_limit, color='gray', linestyle='-.', lw=2)
axs.text(our_data["fa"][-1], planck_limit - 0.025, 'Planck 2018',
         color='gray', fontsize=12, va='bottom', ha='right')

# Shade region above Planck limit
ymin, ymax = axs.get_ylim()
xmin, xmax = axs.get_xlim()
axs.set_xlim(xmin, xmax)
axs.set_ylim(-0.05, ymax)
xaxes = np.linspace(xmin, xmax, num=50)
axs.fill_between(xaxes, planck_limit, ymax, color='gray', alpha=0.3)

# --- Log-scale x-axis tick formatting ---
axs.xaxis.set_major_locator(ticker.LogLocator(base=10.0))
axs.xaxis.set_major_formatter(
    ticker.FuncFormatter(lambda val, pos: f'$10^{{{int(np.log10(val))}}}$')
)
axs.tick_params(axis='x', which='major', labelsize=12)
axs.tick_params(axis='y', which='major', labelsize=12)

# --- Display or save the figure ---
plt.tight_layout()

if flag_save_plot_comparison:
    plt.savefig(f'{proces_name}_our_delta_neff.png', format='png', dpi=300)
    plt.close()
else:
    plt.show()

In [7]:
# --- Log-Log Plot: ΔN_eff — Our Integration vs Maxim's Data ---

fig, ax = plt.subplots(figsize=(8, 6))

# --- Plot Our computed ΔN_eff from distribution integration ---
ax.plot(our_data["fa"], our_data["delta_neff"], color='red', lw=3, linestyle="--",
        label=r'$\Delta N_{\rm eff}$: Our, fBE (integration of distributions)')

# --- Plot Our ΔN_eff results (from second interpolation using Maxim's distribution) ---
ax.plot(our_data_int["fa"], our_data_int["delta_neff"], color='blue', lw=3, linestyle='--',
         label=r'$\Delta N_{\rm eff}$: Our, fBE (second interpolation)')

# --- Set both axes to log scale ---
ax.set_xscale('log')
ax.set_yscale('log')

# --- Axis labels and plot title ---
ax.set_xlabel(r'Decay Constant $\bf f_a$ [GeV]', fontsize=14, fontweight='bold')
ax.set_ylabel(r'$\bf \Delta N_{\rm eff}$', fontsize=14, fontweight='bold')
ax.set_title(title_arr[file_number], fontsize=16, fontweight='bold')

# --- Grid and legend ---
ax.grid(True, linestyle='--', linewidth=0.5)
ax.legend(fontsize=12)

# --- Planck 2018 constraint line and shaded region ---
planck_limit = 0.30
ax.axhline(planck_limit, color='gray', linestyle='-.', lw=2)

# Add text label on the constraint line
ax.text(our_data["fa"][-1], planck_limit * 0.8, 'Planck 2018',
        color='gray', fontsize=12, va='bottom', ha='right')

# Shade region above Planck constraint
ymin, ymax = ax.get_ylim()
xmin, xmax = ax.get_xlim()
ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)
xaxes = np.linspace(xmin, xmax, num=50)
ax.fill_between(xaxes, planck_limit, ymax, color='gray', alpha=0.3)

# --- Log-scale tick formatting on x-axis ---
ax.xaxis.set_major_locator(ticker.LogLocator(base=10.0))
ax.xaxis.set_major_formatter(
    ticker.FuncFormatter(lambda val, pos: f'$10^{{{int(np.log10(val))}}}$')
)
ax.tick_params(axis='x', which='major', labelsize=12)
ax.tick_params(axis='y', which='major', labelsize=12)

# --- Layout and save/display ---
plt.tight_layout()

if flag_save_plot_comparison:
    plt.savefig(f'{proces_name}_our_delta_neff_loglog.png', format='png', dpi=300)
    plt.close()
else:
    plt.show()

In [8]:
# our_data_int["delta_neff"]